# Carga y validación inicial del dataset para predicción del comportamiento de pago

## Objetivo

Este notebook realiza la carga y validación inicial del dataset oficial de
créditos, como primer paso del pipeline de un modelo supervisado orientado
a estimar el comportamiento de pago de nuevos solicitantes de crédito.

En esta etapa **todavía no se realiza análisis exploratorio, Ingeniería de
Características, entrenamiento, evaluación ni generación de
predicciones**. El único propósito es confirmar que el archivo fuente
existe, es legible, y que su estructura, tipos de datos y calidad inicial
cumplen las condiciones mínimas necesarias antes de avanzar a las
siguientes etapas del proyecto (`comprension_eda.ipynb` y la etapa de
Ingeniería de Características).

Concretamente, este notebook cubre:

- Lectura del archivo fuente.
- Validación de la estructura (esquema de columnas).
- Revisión de tipos de datos.
- Revisión de valores nulos.
- Revisión de registros duplicados.
- Validación de consistencia de variables críticas (rangos plausibles).

## Caso de negocio

Una entidad financiera necesita utilizar información histórica de
créditos para construir, en etapas posteriores del proyecto, un modelo
predictivo que permita estimar el comportamiento de pago de nuevos
solicitantes antes del otorgamiento de un crédito. La variable objetivo
del proyecto es `Pago_atiempo`, la cual indica si un cliente cumplió (1) o
no (0) con el pago oportuno de su obligación crediticia.

Antes de construir cualquier modelo o realizar un análisis exploratorio,
es indispensable garantizar que los datos de entrada cumplen condiciones
mínimas de calidad: que el archivo exista, que contenga las variables
esperadas, que los tipos de datos sean coherentes, y que se conozca de
antemano el estado de nulos y duplicados. Omitir esta validación inicial
expondría al proyecto al riesgo de construir un análisis o un modelo sobre
una fuente de datos incompleta, mal estructurada o inconsistente, lo cual
comprometería directamente la confiabilidad de cualquier estimación de
riesgo crediticio derivada de ella.

## Alcance

**Qué sí hace este notebook:**

- Carga el archivo CSV oficial del proyecto.
- Valida que el archivo exista antes de intentar leerlo.
- Valida que las columnas del archivo coincidan con el esquema esperado.
- Reporta dimensiones, tipos de datos, valores nulos y registros
  duplicados.
- Cuantifica cuántos registros violan rangos plausibles definidos para
  variables numéricas críticas (`edad_cliente`, `puntaje`,
  `puntaje_datacredito`).

**Qué NO hace este notebook:**

- No realiza análisis exploratorio de datos (EDA); eso se desarrolla en
  `comprension_eda.ipynb`.
- No imputa valores faltantes.
- No elimina registros ni corrige valores fuera de rango.
- No crea variables nuevas ni aplica transformaciones (Ingeniería de
  Características).
- No entrena ni evalúa ningún modelo.
- No genera predicciones.

Todo hallazgo de calidad detectado aquí se documenta, pero su tratamiento
(corrección, imputación o exclusión) se decide y ejecuta en etapas
posteriores del pipeline.

## Contrato de entrada

- **Archivo:** `Base_de_datos.xlsx - Hoja1.csv`, ubicado un nivel por
  encima de este notebook dentro de la estructura del repositorio.
- **Formato esperado:** archivo CSV delimitado por comas, legible
  mediante `pandas.read_csv`.
- **Variables esperadas:** 22 variables predictoras (crediticias,
  laborales, financieras y de comportamiento de pago) más la variable
  objetivo `Pago_atiempo`, según el esquema definido en
  `EXPECTED_SCHEMA`.
- **Variable objetivo:** `Pago_atiempo`, variable binaria que indica si
  el cliente pagó (1) o no pagó (0) a tiempo.

## Contrato de salida

- Un `DataFrame` `df` cargado y validado en esquema.
- Una tabla de validación de esquema (`schema_validation`), que confirma
  si las columnas coinciden con lo esperado.
- Un resumen de dimensiones (número de registros y de variables).
- Un resumen de tipos de datos por variable.
- Un resumen de valores nulos por variable, con su porcentaje respecto
  al total.
- El conteo de registros duplicados.
- Una tabla de validación de rangos (`range_validation`) que cuantifica
  cuántos registros están fuera de los límites plausibles definidos
  para variables críticas.

Este notebook no persiste ningún archivo intermedio en disco: la
siguiente etapa (`comprension_eda.ipynb`) vuelve a cargar el CSV original
y reutiliza las mismas reglas de validación aquí definidas.

## Flujo del notebook

```text
Carga del CSV
      |
      v
Validacion de estructura (esquema)
      |
      v
Revision de dimensiones
      |
      v
Revision de tipos de datos
      |
      v
Revision de valores nulos
      |
      v
Revision de duplicados
      |
      v
Validacion de rangos plausibles
      |
      v
Diagnostico inicial de calidad
      |
      v
Entrega del dataset validado a comprension_eda.ipynb
```

## Librerías

- **`pandas`**: librería principal del notebook; se utiliza para leer el
  archivo CSV oficial (`pd.read_csv`) y para construir las tablas de
  diagnóstico de calidad (tipos de datos, nulos, duplicados, rangos).
- **`pathlib.Path`**: gestiona la ruta del dataset de forma relativa a la
  ubicación del notebook, evitando rutas absolutas que dependerían del
  entorno local de cada persona que ejecute el proyecto.
- **`logging`**: reemplaza el uso de `print` para registrar el progreso
  de la validación. Esto permite que los mensajes de esta etapa sean
  compatibles con un orquestador de pipeline (Airflow, Prefect, cron) y
  no solo con la ejecución interactiva en Jupyter.
- **`typing.Iterable`**: se utiliza para anotar el tipo de argumento que
  recibe la función de validación de esquema, mejorando la legibilidad y
  el soporte de autocompletado/verificación estática del código.
- **`__future__.annotations`**: habilita evaluación diferida de
  anotaciones de tipo, permitiendo *type hints* más flexibles sin afectar
  el comportamiento en tiempo de ejecución.

In [1]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Iterable

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("riesgo_credito.carga_datos")

## Configuración

La ruta del dataset se define de forma relativa a la ubicación del
notebook. Esta decisión mantiene el notebook portable dentro de la
estructura oficial del proyecto y evita dependencias con rutas absolutas
del equipo local. Los mensajes de salida tampoco exponen la ruta absoluta
resuelta, para no filtrar información del entorno local (usuario, sistema
operativo, estructura de carpetas) en un notebook que puede versionarse.

También se centralizan aquí el esquema esperado del dataset y las reglas
de rango válido para variables numéricas críticas. Estas reglas surgen de
hallazgos concretos de la revisión inicial del dataset: se detectaron
valores imposibles como edades superiores a 120 años, puntajes negativos
y valores de puntaje de central de riesgo fuera del rango típico
(150-950 en el contexto colombiano). Documentar estas reglas en la etapa
de carga permite detectarlas tan pronto como sea posible dentro del
pipeline, antes de que se propaguen a etapas de análisis o modelado.

In [2]:
DATA_PATH = Path("../Base_de_datos.xlsx - Hoja1.csv")
RANDOM_STATE = 42  # constante de reproducibilidad compartida con etapas futuras

# Esquema esperado del dataset.
# Se utilizará para validar que el archivo recibido
# contenga exactamente las variables requeridas, en el orden esperado.
EXPECTED_SCHEMA = [
    "tipo_credito",
    "fecha_prestamo",
    "capital_prestado",
    "plazo_meses",
    "edad_cliente",
    "tipo_laboral",
    "salario_cliente",
    "total_otros_prestamos",
    "cuota_pactada",
    "puntaje",
    "puntaje_datacredito",
    "cant_creditosvigentes",
    "huella_consulta",
    "saldo_mora",
    "saldo_total",
    "saldo_principal",
    "saldo_mora_codeudor",
    "creditos_sectorFinanciero",
    "creditos_sectorCooperativo",
    "creditos_sectorReal",
    "promedio_ingresos_datacredito",
    "tendencia_ingresos",
    "Pago_atiempo",
]

# Reglas de rango plausible para variables numéricas críticas.
# Fuente: hallazgos de la revisión técnica inicial del dataset.
VALUE_RANGE_RULES = {
    "edad_cliente": (18, 100),
    "puntaje": (0, 100),
    "puntaje_datacredito": (150, 950),
}

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

## Funciones del pipeline de carga

La lógica de carga y validación se encapsula en funciones puras, con
docstrings y *type hints*, de modo que puedan reutilizarse desde
`comprension_eda.ipynb` o desde un script de producción sin duplicar
código entre notebooks (principio DRY, requisito básico de un pipeline
MLOps mantenible).

La validación de esquema **detiene la ejecución** (`raise ValueError`) si
el archivo recibido no coincide con el esquema esperado, aplicando el
mismo criterio de "fail fast" que ya rige la validación de existencia del
archivo: es preferible que el pipeline falle de inmediato y de forma
explícita, a que continúe operando sobre una fuente de datos
estructuralmente incorrecta.

In [3]:
def validate_file_exists(path: Path) -> None:
    """Verifica que el archivo fuente exista antes de intentar leerlo.

    Lanza FileNotFoundError si el archivo no está disponible en la ruta
    relativa esperada. No expone la ruta absoluta del sistema local en el
    mensaje de éxito, para mantener el notebook libre de información de
    entorno cuando se versiona.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró el dataset oficial en la ruta relativa: {path}\n"
            f"Directorio de trabajo actual: {Path.cwd()}\n"
            f"Ruta absoluta esperada: {path.resolve()}\n"
            "Verifique que el notebook se ejecute desde la carpeta "
            "'notebooks/' del repositorio y que el archivo "
            "'Base_de_datos.xlsx - Hoja1.csv' exista un nivel arriba de esa "
            "carpeta. Si su estructura de carpetas es distinta, actualice "
            "DATA_PATH en la celda de configuración."
        )
    logger.info("Archivo fuente localizado correctamente en: %s", path)


def load_dataset(path: Path) -> pd.DataFrame:
    """Carga el dataset oficial desde un archivo CSV.

    Aplica validate_file_exists antes de leer, y registra las dimensiones
    resultantes en el log del pipeline.
    """
    validate_file_exists(path)
    df = pd.read_csv(path)
    logger.info(
        "Dataset cargado: %s registros, %s variables.", df.shape[0], df.shape[1]
    )
    return df


def validate_schema(df: pd.DataFrame, expected_schema: Iterable[str]) -> pd.DataFrame:
    """Valida que las columnas del DataFrame coincidan con el esquema oficial.

    Lanza ValueError si existen columnas faltantes o no esperadas. Si las
    columnas coinciden mas no respetan el orden exacto, se registra una
    advertencia (no se considera un error bloqueante, porque el orden de
    columnas de un CSV no siempre está garantizado por la fuente).
    """
    expected_schema = list(expected_schema)
    observed_columns = list(df.columns)
    missing_columns = sorted(set(expected_schema) - set(observed_columns))
    unexpected_columns = sorted(set(observed_columns) - set(expected_schema))
    schema_is_valid = observed_columns == expected_schema

    schema_validation = pd.DataFrame(
        {
            "validacion": [
                "columnas_en_orden_esperado",
                "columnas_faltantes",
                "columnas_no_esperadas",
            ],
            "resultado": [schema_is_valid, missing_columns, unexpected_columns],
        }
    )

    if missing_columns or unexpected_columns:
        raise ValueError(
            f"Esquema inválido. Columnas faltantes: {missing_columns}. "
            f"Columnas no esperadas: {unexpected_columns}."
        )
    if not schema_is_valid:
        logger.warning(
            "Las columnas coinciden pero no respetan el orden esperado del esquema."
        )

    logger.info("Esquema validado correctamente.")
    return schema_validation


def validate_value_ranges(df: pd.DataFrame, rules: dict) -> pd.DataFrame:
    """Cuantifica, por variable, cuántos registros violan un rango plausible.

    No elimina ni corrige valores: esta etapa solo diagnostica y documenta
    el hallazgo para que sea tratado explícitamente en Feature Engineering.
    """
    rows = []
    for column, (lower, upper) in rules.items():
        if column not in df.columns:
            continue
        series = df[column]
        violations = series[(series < lower) | (series > upper)]
        rows.append(
            {
                "variable": column,
                "rango_valido": f"[{lower}, {upper}]",
                "valores_fuera_de_rango": int(violations.shape[0]),
                "porcentaje": round(violations.shape[0] / len(df) * 100, 2),
            }
        )
    return pd.DataFrame(rows).sort_values("valores_fuera_de_rango", ascending=False)

## Ejecución: carga y validación de esquema

Se ejecuta el pipeline de carga y, de forma inmediata, la validación de
esquema. Si el esquema no es válido, la ejecución se detiene aquí mismo:
ninguna celda posterior debería ejecutarse sobre una fuente de datos
estructuralmente incorrecta.

In [4]:
df = load_dataset(DATA_PATH)
schema_validation = validate_schema(df, EXPECTED_SCHEMA)
schema_validation

INFO | Archivo fuente localizado correctamente en: ..\Base_de_datos.xlsx - Hoja1.csv
INFO | Dataset cargado: 10763 registros, 23 variables.
INFO | Esquema validado correctamente.


,validacion,resultado
0,columnas_en_orden_esperado,True
1,columnas_faltantes,[]
2,columnas_no_esperadas,[]


**Interpretación de resultados.** La validación confirma que el archivo
fuente se cargó correctamente y que las columnas observadas coinciden con
el esquema oficial esperado para el proyecto. Para un modelo de riesgo
crediticio, esta validación es la primera línea de defensa: si el esquema
cambiara sin que el equipo de datos lo advirtiera (por ejemplo, una
variable financiera renombrada o eliminada en la fuente), cualquier
análisis o modelo construido a partir de este dataset dejaría de ser
confiable sin que resulte evidente a simple vista. Detener el pipeline de
forma explícita ante un esquema inválido evita que ese riesgo se propague
de forma silenciosa hacia el análisis exploratorio y el modelado.

## Dimensiones del dataset

La validación de dimensiones confirma el volumen inicial de observaciones
y variables que alimentará los siguientes notebooks del proyecto.

In [5]:
dataset_shape = {"registros": df.shape[0], "variables": df.shape[1]}
print(dataset_shape)

{'registros': 10763, 'variables': 23}


**Hallazgo de negocio.** El número de registros disponibles determina la
robustez estadística de cualquier análisis o modelo posterior: una base
con más de 10 mil créditos históricos permite estimar con razonable
confianza patrones de comportamiento de pago, incluso en presencia de un
desbalance de clases entre clientes que pagan a tiempo y clientes que no.

## Información general

La información general permite revisar de forma consolidada la cantidad
de registros no nulos por variable, el tipo inferido por `pandas` y el
uso de memoria del DataFrame.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tipo_credito                   10763 non-null  int64  
 1   fecha_prestamo                 10763 non-null  str    
 2   capital_prestado               10763 non-null  int64  
 3   plazo_meses                    10763 non-null  int64  
 4   edad_cliente                   10763 non-null  int64  
 5   tipo_laboral                   10763 non-null  str    
 6   salario_cliente                10763 non-null  int64  
 7   total_otros_prestamos          10763 non-null  int64  
 8   cuota_pactada                  10763 non-null  int64  
 9   puntaje                        10763 non-null  float64
 10  puntaje_datacredito            10757 non-null  float64
 11  cant_creditosvigentes          10763 non-null  int64  
 12  huella_consulta                10763 non-null  int64  
 1

**Interpretación de resultados.** Este resumen permite detectar de un
vistazo si alguna variable presenta una cantidad de valores no nulos
notoriamente inferior al total de registros, lo cual sería una primera
señal de incompletitud del historial crediticio de un segmento de
clientes. Esa señal se cuantifica en detalle en la siguiente sección.

## Tipos de datos

La inspección de tipos permite identificar variables numéricas, variables
textuales y posibles campos que requieran conversión en etapas
posteriores. En este notebook solo se reportan los tipos inferidos; no se
transforman columnas.

In [7]:
data_types = (
    df.dtypes.astype(str)
    .rename("tipo_dato")
    .reset_index()
    .rename(columns={"index": "variable"})
)
data_types

,variable,tipo_dato
0,tipo_credito,int64
1,fecha_prestamo,str
2,capital_prestado,int64
3,plazo_meses,int64
4,edad_cliente,int64
5,tipo_laboral,str
6,salario_cliente,int64
7,total_otros_prestamos,int64
8,cuota_pactada,int64
9,puntaje,float64


**Interpretación de resultados.** Confirmar el tipo de dato inferido por
`pandas` para cada variable es un paso necesario antes de cualquier
modelado: una variable financiera leída como texto en lugar de numérica,
o una fecha leída como texto plano, pasaría desapercibida en el análisis
exploratorio y podría excluirse por error del conjunto de variables
predictoras utilizables por el modelo de riesgo crediticio.

## Valores nulos

El conteo de valores nulos permite dimensionar la completitud del dataset
antes del análisis exploratorio. La definición de tratamiento de nulos se
reserva para etapas posteriores; aquí solo se documenta la situación
inicial.

In [8]:
missing_values = (
    df.isna()
    .sum()
    .rename("nulos")
    .reset_index()
    .rename(columns={"index": "variable"})
)
missing_values["porcentaje"] = (missing_values["nulos"] / len(df) * 100).round(2)
missing_values = (
    missing_values.query("nulos > 0")
    .sort_values("nulos", ascending=False)
    .reset_index(drop=True)
)
missing_values

,variable,nulos,porcentaje
0,tendencia_ingresos,2932,27.24
1,promedio_ingresos_datacredito,2930,27.22
2,saldo_mora_codeudor,590,5.48
3,saldo_principal,405,3.76
4,saldo_mora,156,1.45
5,saldo_total,156,1.45
6,puntaje_datacredito,6,0.06


**Hallazgo de negocio.** La presencia de valores faltantes en variables
financieras o de historial crediticio puede representar información
incompleta del perfil del cliente —por ejemplo, solicitantes sin
historial suficiente en una central de riesgo— y no necesariamente un
error de captura. Si el porcentaje de nulos es alto en variables con
relevancia para el riesgo crediticio, esto puede afectar la capacidad
predictiva del modelo y debe tratarse con una estrategia de imputación
explícita y justificada en la etapa de Ingeniería de Características, en
lugar de eliminarse sin análisis.

## Registros duplicados

La revisión de duplicados identifica si existen filas completamente
repetidas. Este control es una validación de calidad; no implica
eliminación de registros en este notebook.

In [9]:
duplicate_rows = int(df.duplicated().sum())
print({"registros_duplicados": duplicate_rows})

{'registros_duplicados': 0}


**Interpretación de resultados.** La presencia de registros duplicados
podría distorsionar el aprendizaje de un modelo supervisado, ya que un
mismo caso de comportamiento de pago se repetiría con mayor peso relativo
del que le corresponde. Confirmar su ausencia (o cuantificarlos, si
existieran) en esta etapa evita que ese sesgo se introduzca de forma
silenciosa en el análisis exploratorio o en el entrenamiento del modelo.

## Validación de rangos válidos

Además de la validación estructural (esquema), se valida que un
subconjunto de variables numéricas críticas se encuentre dentro de un
rango plausible según el negocio: `edad_cliente` no debería superar los
100 años, `puntaje` está definido en una escala de 0 a 100, y
`puntaje_datacredito` corresponde a un score de central de riesgo cuyo
rango típico en Colombia es 150-950. Esta validación **no elimina ni
corrige** registros; únicamente cuantifica el hallazgo para que se trate
explícitamente en la etapa de comprensión exploratoria y en Ingeniería de
Características. No tratar esto en silencio evita que valores imposibles
se interpreten más adelante como señales estadísticas válidas del
comportamiento de pago.

In [10]:
range_validation = validate_value_ranges(df, VALUE_RANGE_RULES)
range_validation

,variable,rango_valido,valores_fuera_de_rango,porcentaje
2,puntaje_datacredito,"[150, 950]",153,1.42
0,edad_cliente,"[18, 100]",150,1.39
1,puntaje,"[0, 100]",135,1.25


**Hallazgo de negocio.** Un valor de edad superior a 100 años, un
puntaje negativo o un score de central de riesgo fuera de su rango típico
no representan variabilidad legítima del negocio crediticio: son errores
de captura o de codificación en la fuente de datos. Si estos registros no
se identifican y tratan antes del modelado, podrían distorsionar la
estimación del riesgo asociado a variables clave como el score de central
de riesgo, uno de los indicadores más directamente relacionados con el
comportamiento de pago de un cliente.

## Buenas prácticas MLOps aplicadas en este notebook

- **Reproducibilidad:** la ruta del dataset se define de forma relativa,
  y el esquema y las reglas de rango se centralizan como constantes, de
  modo que el notebook produzca el mismo resultado en cualquier entorno
  que respete la estructura del repositorio.
- **Trazabilidad:** el uso de `logging` en lugar de `print` deja un
  registro estructurado y con nivel de severidad (`INFO`, `WARNING`) de
  cada paso de la validación, compatible con su ejecución dentro de un
  orquestador de pipeline.
- **Calidad del dato:** la validación de esquema, tipos, nulos,
  duplicados y rangos se ejecuta de forma sistemática y documentada antes
  de avanzar a cualquier etapa de análisis o modelado, evitando que
  problemas de calidad se propaguen de forma silenciosa.
- **Modularidad:** la lógica de validación se encapsula en funciones
  puras y reutilizables (`validate_file_exists`, `load_dataset`,
  `validate_schema`, `validate_value_ranges`), que pueden invocarse desde
  otros notebooks o scripts del proyecto sin duplicar código.
- **Versionamiento:** al no exponer rutas absolutas ni información del
  entorno local en los mensajes de salida, el notebook puede versionarse
  en un repositorio compartido sin filtrar detalles del equipo de cada
  desarrollador.
- **Contrato de pipeline:** al declarar explícitamente qué recibe
  (contrato de entrada) y qué entrega (contrato de salida) este
  notebook, se establece una interfaz clara con la siguiente etapa del
  proyecto (`comprension_eda.ipynb`), reduciendo el riesgo de
  suposiciones implícitas entre etapas.

## Conclusiones

La validación inicial confirma que el dataset oficial está disponible y
se carga correctamente con `pandas`, utilizando una ruta relativa que
preserva la portabilidad del proyecto dentro de la estructura del
repositorio, sin exponer rutas absolutas del entorno local en los
mensajes de salida.

El conjunto de datos está conformado por **10.763 registros** y **23
variables**, sin evidencia de registros completamente duplicados. La
validación de esquema, de carácter bloqueante, confirma que todas las
columnas esperadas están presentes; si en una ejecución futura el
esquema cambiara, el pipeline se detendrá de forma explícita en esta
celda en lugar de propagar una fuente de datos inconsistente a las
siguientes etapas.

Se identificaron valores faltantes concentrados principalmente en
`tendencia_ingresos` y `promedio_ingresos_datacredito` (ambas alrededor
de 27%), y en menor medida en variables de saldo y mora. La coincidencia
de magnitud entre las dos primeras variables sugiere un origen común que
debe investigarse en la etapa de comprensión exploratoria (por ejemplo,
un segmento de clientes sin historial suficiente en una central de
riesgo).

La validación de rangos detectó valores fuera de los límites plausibles
en `edad_cliente`, `puntaje` y `puntaje_datacredito`. Estos casos no se
corrigen en esta etapa: quedan documentados como hallazgos de calidad de
datos que deben tratarse explícitamente antes del modelado, ya sea como
errores de captura a corregir o como reglas de imputación específicas.

Finalmente, se verifica que la variable objetivo disponible corresponde a
**`Pago_atiempo`**, por lo que el proyecto utilizará esta columna como
referencia para las fases posteriores de análisis, ingeniería de
características y modelado predictivo del comportamiento de pago.